# Parse market
Fold sorted parsed FIX messages into books, or write their orders and executions directly.

In [ ]:
source = "fixmessage.market"
books = True
target = "market.books"
order_target = "market.orders"
execution_target = "market.executions"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
branch = "root"
snapshot_every = 3_600_000_000_000
max_lateness_ns = 900_000_000_000
max_order_age_ns = 86_400_000_000_000
max_side_alive = 10_000
merge_by = True
commit_row_size = 250_000

In [ ]:
import pyarrow.compute as pc
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    LessThan,
    LessThanOrEqual,
    NotEqualTo,
    NotNull,
)
from rekep.enums import EventType
from rekep.iceberg import IcebergDataset
from rekep.market import Book, BookIterator, Execution, Order
from rekep.text import FixMessage
from rekep.times import unix_of

HOUR = 3_600_000_000_000
DAY = 86_400_000_000_000


configured_targets = (target,) if books else tuple(
    name for name in (order_target, execution_target) if name is not None
)
if not configured_targets or configured_targets[0] is None:
    raise ValueError("book mode needs target; direct mode needs an event target")
if len(configured_targets) != len(set(configured_targets)):
    raise ValueError("market targets must be distinct")


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


lower, upper = unix_of(start), unix_of(end, upper=True)
read_lower = None if lower is None else lower - lower % HOUR - HOUR
read_upper = None if upper is None else ((upper + HOUR - 1) // HOUR) * HOUR + max_lateness_ns
logs_table = IcebergDataset(
    name=source, catalog=catalog, properties=dict(catalog_properties), branch=branch
)
book_table = None if not books else IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
    field=Book.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "hash"),
)


def _event_table(name, event_type):
    if name is None:
        return None
    return IcebergDataset(
        name=name,
        catalog=catalog,
        properties=dict(catalog_properties),
        branch=branch,
        field=event_type.into_field(),
        commit_row_size=commit_row_size,
        sort_by=("unix", "hash"),
    )


order_table = None if books else _event_table(order_target, Order)
execution_table = None if books else _event_table(execution_target, Execution)


def _book_seeds():
    if book_table is None or read_lower is None:
        return ()
    recent = And(
        GreaterThanOrEqual("unix", read_lower - DAY),
        LessThanOrEqual("unix", read_lower),
        NotNull("sunix"),
    )
    reader = book_table.read_arrow_reader(
        Book.into_field(), row_filter=recent, order_by=("unix", "hash")
    )
    return Book.from_arrow_reader(reader)


def _log_reader():
    row_filter = _window(read_lower, read_upper)
    if book_table is None:
        market_events = NotEqualTo("etype", int(EventType.INSTRUMENT))
        row_filter = market_events if row_filter is None else And(row_filter, market_events)
    reader = logs_table.read_arrow_reader(
        FixMessage.into_field(),
        row_filter=row_filter,
        order_by=("unix", "msg_seq_num", "hash"),
    )
    return reader


def _logs():
    return FixMessage.from_arrow_reader(_log_reader())


def _inside(event):
    return (lower is None or event.unix >= lower) and (upper is None or event.unix < upper)


def _write_books():
    snapshots = _book_seeds()
    read = {"books": 0, "orders": 0, "executions": 0}
    iterating = BookIterator(
        logs=_logs(),
        snapshots=snapshots,
        snapshot_every=snapshot_every,
        snapshot_until=upper,
        max_order_age_ns=max_order_age_ns,
        max_side_alive=max_side_alive,
    )

    def selected():
        nonlocal read
        for book in iterating:
            if _inside(book):
                read["books"] += 1
                read["orders"] += len(book.deltas)
                read["executions"] += len(book.executions)
                yield book

    written = book_table.append_arrow_reader(
        Book.into_arrow_reader(selected()),
        Book.into_field(),
        merge_by=merge_by,
        commit_row_size=commit_row_size,
    )
    checkpoint = min((book.unix for book in iterating.snapshots), default=None)
    return read, written, checkpoint


def _write_events():
    tables = {Order: order_table, Execution: execution_table}
    read = {Order: 0, Execution: 0}
    written = {Order: 0, Execution: 0}
    batch_rows = None if commit_row_size == 0 else commit_row_size or 65_536
    for event_type, batch in FixMessage.into_market_arrow_batches(
        _log_reader(), batch_row_size=batch_rows
    ):
        mask = None
        if lower is not None:
            mask = pc.greater_equal(batch.column("unix"), lower)
        if upper is not None:
            before = pc.less(batch.column("unix"), upper)
            mask = before if mask is None else pc.and_(mask, before)
        if mask is not None:
            batch = batch.filter(mask)
        read[event_type] += batch.num_rows
        table = tables[event_type]
        if table is None or not batch.num_rows:
            continue
        written[event_type] += table.append_arrow_reader(
            iter((batch,)),
            event_type.into_field(),
            merge_by=merge_by,
            commit_row_size=commit_row_size,
        )
    return read, written


read = {"books": 0, "orders": 0, "executions": 0}
written = dict(read)
flatten = {"orders": 0, "executions": 0}
checkpoint = None
if book_table is not None:
    book_read, written["books"], checkpoint = _write_books()
    read.update(book_read)
    flatten.update({name: book_read[name] for name in flatten})
else:
    event_read, event_written = _write_events()
    read["orders"], read["executions"] = event_read[Order], event_read[Execution]
    written["orders"], written["executions"] = (
        event_written[Order],
        event_written[Execution],
    )
result = {
    "mode": "books" if book_table is not None else "events",
    "read": read,
    "written": written,
    "flatten": flatten,
    "checkpoint": checkpoint,
    "read_lower": read_lower,
    "read_upper": read_upper,
    "targets": {
        "books": target if books else None,
        "orders": None if books else order_target,
        "executions": None if books else execution_target,
    },
}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result